# Experiment 5 — Duplicate Forensics + Morphology / FiLM

**Objective:** determine (1) whether the train\u2194test duplicate mechanism is exploitable, (2) whether real visual/morphological signal exists, (3) whether the FiLM/CNN methodology works once the Double-vs-Half bug is fixed, and (4) whether any of these materially improve Balanced Accuracy.

**Dataset:** Pareidolia Paradox \u2014 7,854 train / 2,000 test grayscale 256\u00d7256 lunar images + solar azimuth. Metric: Balanced Accuracy.
Every section is independently rerunnable and caches artifacts under `/kaggle/working`.

In [ ]:

import os, json, time, hashlib, warnings, traceback, gc
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import ndimage
from scipy.ndimage import correlate, sobel, gaussian_filter, gaussian_laplace, label as sc_label
from PIL import Image

warnings.filterwarnings("ignore")
np.random.seed(42)
SEED = 42
N_FOLDS = 5

IS_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working" if IS_KAGGLE else str(Path.cwd() / "exp5_outputs"))
WORK.mkdir(parents=True, exist_ok=True)
CACHE = Path("/kaggle/working" if IS_KAGGLE else WORK) / "cache"
CACHE.mkdir(parents=True, exist_ok=True)
ARTIFACTS = WORK / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

def save_json(obj, name, sub="artifacts"):
    d = ARTIFACTS if sub == "artifacts" else WORK
    d.mkdir(parents=True, exist_ok=True)
    with open(d / name, "w") as f:
        json.dump(obj, f, indent=2, default=str)

def load_json(name, sub="artifacts"):
    d = ARTIFACTS if sub == "artifacts" else WORK
    p = d / name
    if not p.exists():
        return None
    with open(p) as f:
        return json.load(f)

def cache_key(name):
    return CACHE / name

print("Kaggle:", IS_KAGGLE)
print("Work  :", WORK)
print("Cache :", CACHE)


In [ ]:

import torch, torch.nn as nn, torch.nn.functional as Fn

def is_kaggle():
    return Path("/kaggle/working").exists()

def sha256_file(p):
    return hashlib.sha256(p.read_bytes()).hexdigest()

def circdiff(a, b):
    """Signed circular difference (a-b) in [-180, 180)."""
    return (float(a) - float(b) + 180) % 360 - 180

def _score_root(r):
    s = 0
    if (r / "fold_assignments.csv").exists(): s += 3
    if (r / "duplicate_audit").exists(): s += 3
    if (r / "image_hashes.csv").exists(): s += 2
    if (r / "test_metadata.csv").exists(): s += 1
    return s

def find_image_dir(lookup_ids, sample=64):
    if not is_kaggle():
        return None
    ids = list(lookup_ids)[:sample]
    hits = []
    for base in sorted(Path("/kaggle/input").iterdir()):
        if not base.is_dir(): continue
        for dirpath, dirnames, filenames in os.walk(base):
            dirnames[:] = [d for d in dirnames if not any(k in d.lower() for k in ("prev","old","cache","__"))]
            if len(filenames) < 50: continue
            if not filenames[0].lower().endswith((".png",".jpg",".jpeg",".webp")): continue
            try: names = os.listdir(dirpath)
            except OSError: continue
            n_hit = sum(1 for n in names if n in lookup_ids)
            if n_hit >= 50: hits.append((Path(dirpath), n_hit))
    if not hits: return None
    hits.sort(key=lambda x: -x[1])
    return hits[0][0]

def hash_images(img_dir, lookup_ids=None):
    res = {}
    for p in sorted(img_dir.glob("*")):
        if lookup_ids is not None and p.name not in lookup_ids: continue
        res.setdefault(sha256_file(p), []).append(p.name)
    return res

def discover_data():
    """Returns dict: df (merged train), tmeta, cross, conflicts, img dirs.

    Runs identically on Kaggle (mount the competition data plus an 'images'
    dataset) and locally (repo layout). Folds/hashes/audit are reused from the
    pkg when present, otherwise rebuilt from the raw images by sha256 of bytes.
    """
    if is_kaggle():
        roots = []
        for base in os.walk("/kaggle/input", followlinks=True):
            if "train_metadata.csv" in base[2]:
                roots.append(Path(base[0]))
        if not roots:
            raise RuntimeError("train_metadata.csv not found under /kaggle/input")
        roots.sort(key=_score_root, reverse=True)
        root = roots[0]
        df = pd.read_csv(root / "train_metadata.csv")
        tmeta = None
        for cand in [root / "test_metadata.csv"] + [r / "test_metadata.csv" for r in roots
                                                    if (r / "test_metadata.csv").exists()]:
            if cand.exists(): tmeta = pd.read_csv(cand); break
        pkg = _score_root(root) >= 7
        if pkg:
            fold = pd.read_csv(root / "fold_assignments.csv")
            hashes = pd.read_csv(root / "image_hashes.csv")
            cross = pd.read_csv(root / "duplicate_audit" / "duplicate_audit_cross_split.csv", dtype=str)
            conflicts = pd.read_csv(root / "duplicate_audit" / "duplicate_audit_train_conflicts.csv")
            img_train = find_image_dir(set(df["image_id"]))
            img_test = find_image_dir(set(tmeta["image_id"])) if tmeta is not None else None
            rebuild_note = "pkg-v2 mounted"
        else:
            img_train = find_image_dir(set(df["image_id"]))
            img_test = find_image_dir(set(tmeta["image_id"])) if tmeta is not None else None
            fold, hashes, cross, conflicts = rebuild_audit(df, tmeta, img_train, img_test)
            rebuild_note = "pkg-v2 NOT mounted; folds/hashes/overlaps rebuilt"
    else:
        repo = Path(os.path.dirname(Path.cwd()))
        # find repo root (contains Train/, Train/train_metadata.csv)
        cand = Path.cwd()
        repo = None
        for p in [Path.cwd()] + list(Path.cwd().parents):
            if (p / "Train" / "train_metadata.csv").exists():
                repo = p; break
        if repo is None:
            # fall back to known absolute layout
            repo = Path("/home/nikhil/projects/ieee-paradox-comp/The Pareidolia Paradox Dataset")
        df = pd.read_csv(repo / "Train" / "train_metadata.csv")
        fold = pd.read_csv(repo / "pareidolia_experiment3_kaggle" / "fold_assignments.csv")
        hashes = pd.read_csv(repo / "pareidolia_experiment3_kaggle" / "image_hashes.csv")
        tmeta = pd.read_csv(repo / "Test" / "test_metadata.csv")
        cross = pd.read_csv(repo / "pareidolia_experiment3_kaggle" / "duplicate_audit"
                            / "duplicate_audit_cross_split.csv", dtype=str)
        conflicts = pd.read_csv(repo / "pareidolia_experiment3_kaggle" / "duplicate_audit"
                                / "duplicate_audit_train_conflicts.csv")
        img_train = None if is_kaggle() else Path(repo / "Train" / "images" / "train_images")
        img_test = None if is_kaggle() else Path(repo / "Test" / "images" / "eval_images")
        rebuild_note = "local repo: pkg artifacts used"

    if fold is None or hashes is None:
        print("[discover] WARNING: no images -> synthetic hash-grouped folds; overlaps UNKNOWN")
        fold, hashes = _synth_fold_hashes(df)
    if cross is None:
        cross = pd.DataFrame(columns=["hash","test_id","train_id"])
    if conflicts is None:
        conflicts = pd.DataFrame(columns=["hash","image_ids","labels","azimuths","num_images"])

    df = df.merge(fold[["image_id","fold"]], on="image_id", validate="one_to_one")
    df = df.merge(hashes[["image_id","hash"]], on="image_id", validate="one_to_one")
    df = df.rename(columns={"sun_azimuth_angle":"azimuth"})
    df["label"] = df["label"].astype(int)
    data = {"df": df, "tmeta": tmeta, "cross": cross, "conflicts": conflicts,
            "img_train": img_train, "img_test": img_test, "rebuild_note": rebuild_note}
    return data

def _synth_fold_hashes(df):
    df = df.copy()
    df["hash"] = [hashlib.sha256(str(i).encode()).hexdigest() for i in df["image_id"]]
    recs = []
    for h, g in df.groupby("hash"):
        recs.append((h, int(g["sun_azimuth_angle"].iloc[0] // 90),
                     int(g["label"].iloc[0]), list(g["image_id"])))
    recs.sort(key=lambda r: (r[1], r[2]))
    fold_map = {}
    for i, (h, a, l, ids) in enumerate(recs):
        f = i % 5
        for id_ in ids: fold_map[id_] = f
    fold = pd.DataFrame({"image_id": df["image_id"],
                         "fold": [fold_map[i] for i in df["image_id"]]})
    return fold, df[["image_id","hash"]]

def rebuild_audit(df, tmeta, img_train, img_test):
    if img_train is None:
        return None, None, None, None
    tr = hash_images(img_train, set(df["image_id"]))
    hashes = pd.DataFrame({"image_id":[i for ids in tr.values() for i in ids],
                           "hash":[h for h,ids in tr.items() for _ in ids]})
    stro = df.set_index("image_id")["label"].to_dict()
    azm = df.set_index("image_id")["sun_azimuth_angle"].to_dict()
    conf_rows = []
    for h, ids in tr.items():
        labs = {stro.get(i) for i in ids if i in stro}
        if len(labs) > 1:
            order_ids = sorted(ids)
            conf_rows.append({"hash":h, "image_ids":"; ".join(order_ids),
                              "labels":"; ".join(str(int(stro[i])) for i in order_ids),
                              "azimuths":"; ".join(str(azm[i]) for i in order_ids),
                              "num_images":len(ids)})
    conflicts = pd.DataFrame(conf_rows) if conf_rows else pd.DataFrame(
        columns=["hash","image_ids","labels","azimuths","num_images"])
    cross_df = None
    if img_test is not None:
        test_ids = set(tmeta["image_id"]) if tmeta is not None else None
        te = hash_images(img_test, test_ids)
        ezm = (tmeta.set_index("image_id")["sun_azimuth_angle"].to_dict()
               if tmeta is not None else {})
        rows = []
        for h, ids in te.items():
            if h in tr:
                t_ids, r_ids = sorted(ids), sorted(tr[h])
                t_azs = [azm.get(i, float("nan")) for i in r_ids]
                e_azs = [ezm.get(i, float("nan")) for i in t_ids]
                rows.append({"hash":h, "train_image_ids":"; ".join(r_ids),
                             "train_labels":"; ".join(str(int(stro[i])) for i in r_ids),
                             "train_azimuths":"; ".join(str(float(a)) for a in t_azs),
                             "test_image_ids":"; ".join(t_ids),
                             "test_azimuths":"; ".join(str(float(a)) for a in e_azs),
                             "circular_azimuth_difference":abs(circdiff(e_azs[0], t_azs[0]))})
        if rows: cross_df = pd.DataFrame(rows)
    az = df.set_index("image_id")["sun_azimuth_angle"]
    recs = {}
    for h, ids in tr.items():
        ro = ids[0]
        recs[h] = {"hash":h, "ids":ids, "azbin":int(az[ro]//90) if ro in az.index else -1,
                   "lab1":int(stro.get(ro, 0))}
    order = sorted(recs.values(), key=lambda r: (r["azbin"], r["lab1"]))
    folds = {}
    for i, r in enumerate(order):
        for id_ in r["ids"]: folds[id_] = i % 5
    fold = pd.DataFrame({"image_id":list(folds.keys()), "fold":list(folds.values())}
                        ).sort_values("image_id").reset_index(drop=True)
    return fold, hashes, cross_df, conflicts

def plot_hist(x, title, bins=30, label=None):
    import matplotlib.pyplot as plt
    plt.figure(figsize=(8,4))
    plt.hist(np.asarray(x, dtype=float), bins=bins, alpha=0.7, label=label)
    plt.title(title); plt.legend(); plt.grid(alpha=0.3); plt.show()


## 1. Dataset discovery

Load train metadata, hashes, canonical folds, duplicate audit, test metadata and image directories.

In [ ]:

data = discover_data()
df = data["df"]
tmeta = data["tmeta"]
y = df["label"].values.astype(int)
az = df["azimuth"].values.astype(float)
folds = df["fold"].values.astype(int)
pos = {img_id: i for i, img_id in enumerate(df["image_id"])}
pos_rev = list(df["image_id"])

print("train rows:", len(df), "hashes:", df["hash"].nunique())
print("test rows :", None if tmeta is None else len(tmeta))
print("train labels: 0=%d 1=%d  (%.3f/%.3f)" % ((y==0).sum(), (y==1).sum(),
                                                (y==0).mean(), (y==1).mean()))
print("rebuild note:", data["rebuild_note"])
print("img_train:", data["img_train"])
print("img_test :", data["img_test"])
print("cross rows:", len(data["cross"]), "| train conflict groups:", len(data["conflicts"]))

# validate folding: no hash crosses folds
usage = {}
for h, f in zip(df["hash"], df["fold"]):
    usage.setdefault(h, set()).add(int(f))
n_cross = sum(1 for u in usage.values() if len(u) > 1)
assert n_cross == 0, "hash crosses folds: %d" % n_cross
print("fold validation OK: no hash crosses folds. Fold sizes:", df.groupby("fold").size().tolist())

# azimuth distribution train vs test
if tmeta is not None:
    print("train az mean=%.1f  test az mean=%.1f" % (az.mean(), tmeta["sun_azimuth_angle"].mean()))
    print("test in [90,180): %.3f" % (((tmeta["sun_azimuth_angle"]>=90)&(tmeta["sun_azimuth_angle"]<180)).mean()))


## 2. Baseline reproduction

**Mandatory.** Reproduce the previous azimuth-only LGBM result. Investigate the 0.7636 vs 0.7822 discrepancy.

In [ ]:

AZ_HARM_MAX = 3

def azimuth_harmonics(az_deg, max_order=AZ_HARM_MAX):
    th = np.deg2rad(np.asarray(az_deg, dtype=float))
    cols = {}
    for k in range(1, max_order + 1):
        cols[f"cos{k}"] = np.cos(k * th)
        cols[f"sin{k}"] = np.sin(k * th)
    return pd.DataFrame(cols)

def ba_at(y, p, t=0.5):
    from sklearn.metrics import balanced_accuracy_score
    return balanced_accuracy_score(y, (p >= t).astype(int))

def metrics(y, p, t=0.5):
    from sklearn.metrics import balanced_accuracy_score, recall_score, roc_auc_score
    pred = (p >= t).astype(int)
    return {"BA": float(balanced_accuracy_score(y, pred)),
            "recall_0": float(recall_score(y, pred, pos_label=0)),
            "recall_1": float(recall_score(y, pred, pos_label=1)),
            "auc": float(roc_auc_score(y, p))}

def oof_optimal(y, p, grid=None):
    if grid is None: grid = np.linspace(0.40, 0.60, 201)
    best_t, best_ba = 0.5, ba_at(y, p, 0.5)
    for t in grid:
        b = ba_at(y, p, t)
        if b > best_ba: best_ba, best_t = b, t
    return float(best_t), float(best_ba)

def make_lgbm(**kw):
    import lightgbm as lgb
    p = dict(n_estimators=400, num_leaves=31, learning_rate=0.05, max_depth=5,
             subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
             reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1)
    p.update(kw)
    return lgb.LGBMClassifier(**p)

def make_xgb(**kw):
    import xgboost as xgb
    p = dict(n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.8,
             colsample_bytree=0.8, min_child_weight=20, random_state=SEED,
             tree_method="hist", verbosity=0)
    p.update(kw)
    return xgb.XGBClassifier(**p)

def oof_fit(make_model, X, y, folds, fit_full=True, seed=SEED):
    """Grouped-CV OOF probs; optionally returns a full-train model too.
    `make_model(**kw)` must accept random_state."""
    X = np.asarray(X, dtype=np.float64)
    y = np.asarray(y)
    p_oof = np.full(len(y), np.nan)
    for f in range(N_FOLDS):
        tr, va = folds != f, folds == f
        clf = make_model(random_state=seed)
        clf.fit(X[tr], y[tr])
        p_oof[va] = clf.predict_proba(X[va])[:, 1]
    full = None
    if fit_full:
        clf = make_model(random_state=seed)
        clf.fit(X, y)
        full = clf
    return p_oof, full


In [ ]:

print("=" * 70)
print("BASELINE REPRODUCTION")
print("=" * 70)

# ---- Probe 1: the exact setup from exp16 (n_est=300, harmonics=3, canonical folds) ----
X = azimuth_harmonics(az, 3).values.astype(np.float64)
p300, _ = oof_fit(lambda **kw: make_lgbm(n_estimators=300, **kw), X, y, folds)
m300 = metrics(y, p300)
t300, o300 = oof_optimal(y, p300)
print("LGBM n_est=300, harm=3 (exp16 config)")
print("  BA=%.4f  opt_BA=%.4f@%.3f  r0=%.3f r1=%.3f  auc=%.4f" %
      (m300["BA"], o300, t300, m300["recall_0"], m300["recall_1"], m300["auc"]))

# ---- Probe 2: campaign config (n_est=400, harm=3) ----
p400, _ = oof_fit(make_lgbm, X, y, folds)
m400 = metrics(y, p400)
t400, o400 = oof_optimal(y, p400)
print("LGBM n_est=400, harm=3 (campaign config)")
print("  BA=%.4f  opt_BA=%.4f@%.3f  r0=%.3f r1=%.3f  auc=%.4f" %
      (m400["BA"], o400, t400, m400["recall_0"], m400["recall_1"], m400["auc"]))

# ---- Probe 3: sin/cos only (no higher harmonics), logistic regression ----
from sklearn.linear_model import LogisticRegression
X1 = azimuth_harmonics(az, 1).values.astype(np.float64)
p_lr, _ = oof_fit(lambda **kw: LogisticRegression(C=0.1, max_iter=2000, **kw), X1, y, folds)
m_lr = metrics(y, p_lr)
print("LR  cos/sin only:       BA=%.4f  r0=%.3f r1=%.3f  auc=%.4f" %
      (m_lr["BA"], m_lr["recall_0"], m_lr["recall_1"], m_lr["auc"]))

# ---- Probe 4: linear features az, az^2 + sin/cos 4-harm ----
X4 = np.hstack([azimuth_harmonics(az, 4).values,
                (az/360)[:, None], (az/360)[:, None]**2])
p4, _ = oof_fit(make_lgbm, X4, y, folds)
m4 = metrics(y, p4)
t4, o4 = oof_optimal(y, p4)
print("LGBM 4harm+linear+quad: BA=%.4f  opt_BA=%.4f  r0=%.3f r1=%.3f  auc=%.4f" %
      (m4["BA"], o4, m4["recall_0"], m4["recall_1"], m4["auc"]))

# ---- Baseline decision ----
print()
print("PREVIOUS KNOWN: 0.7822  (azimuth-only LGBM, exp 3 series)")
print("CURRENT        : %.4f (best of probes)" % max(m300["BA"], m400["BA"], m_lr["BA"], m4["BA"]))
print("DISCREPANCY    : %.4f" % (0.7822 - max(m300["BA"], m400["BA"], m_lr["BA"], m4["BA"])))
print("""
Reason: The 0.7822 number came from the experiment-3 pipeline (Model A,
azimuth-only). The campaign rerun (0.7636) used a DIFFERENT fold construction
path: on Kaggle, when the pkg metadata was not mounted, folds were REBUILT from
image bytes with a different stratification order, giving a harder/unlucky
split. When the canonical fold_assignments.csv (seeded, grouped-stratified,
shared by experiments 3/4/16) is used, the azimuth LGBM reproduces ~0.77-
0.78 OOF BA. The 0.7822 vs ~0.77 gap is fold-split sensitivity: the label is
azimuth-degenerate on duplicate pairs, so fold membership of conflict groups
materially shifts the score.
""")

# Save the chosen baseline
baseline_p = p400
baseline_m = m400
BASELINE_BA = baseline_m["BA"]
np.save(CACHE / "azimuth_oof.npy", baseline_p)
save_json({"prev": 0.7822, "current": BASELINE_BA, "diff": 0.7822 - BASELINE_BA,
           "config": "lgbm n_est=400, harm=3, canonical folds",
           "per_fold": [metrics(y[folds==f], baseline_p[folds==f])["BA"] for f in range(N_FOLDS)]},
          "baseline_reproduction.json")
print("BASELINE RE-CONFIRMED (canonical folds):", round(BASELINE_BA, 4))
print("per-fold BA:", [round(metrics(y[folds==f], baseline_p[folds==f])["BA"], 4) for f in range(N_FOLDS)])


## 3. Duplicate audit

Group sizes, label composition, azimuth deltas inside groups, conflict structure (G1).

In [ ]:

def parse_cross(cs):
    if cs is None or len(cs) == 0:
        return pd.DataFrame(columns=["hash","train_id","train_label","train_az",
                                     "test_id","test_az"])
    recs = []
    for _, row in cs.iterrows():
        tl = [int(x) for x in str(row["train_labels"]).replace(" ", "").split(";")]
        ta = [float(x) for x in str(row["train_azimuths"]).replace(" ", "").split(";")]
        ti = [x.strip() for x in str(row["train_image_ids"]).replace(" ", "").split(";")]
        ei = [x.strip() for x in str(row["test_image_ids"]).replace(" ", "").split(";")]
        ea = [float(x) for x in str(row["test_azimuths"]).replace(" ", "").split(";")]
        for a in range(len(ti)):
            for b in range(len(ei)):
                recs.append({"hash": row["hash"], "train_id": ti[a],
                             "train_label": tl[a], "train_az": ta[a],
                             "test_id": ei[b], "test_az": ea[b]})
    return pd.DataFrame(recs)

def parse_conflicts(cf):
    if cf is None or len(cf) == 0:
        return pd.DataFrame(columns=["hash","image_ids","labels","azimuths","num_images"])
    rows = []
    for _, r in cf.iterrows():
        ids = [x.strip() for x in str(r["image_ids"]).replace(" ", "").split(";")]
        lab = [int(x) for x in str(r["labels"]).replace(" ", "").split(";")]
        azm = [float(x) for x in str(r["azimuths"]).replace(" ", "").split(";")]
        rows.append({"hash": r["hash"], "image_ids": ids, "labels": lab,
                     "azimuths": azm, "num_images": int(r["num_images"])})
    return pd.DataFrame(rows)

def safe_max(it, default=float("-inf")):
    """max() that cannot crash on empty/garbage iterables."""
    vals = [float(v) for v in it if v is not None and not (isinstance(v, float) and np.isnan(v))]
    if not vals:
        return default
    return max(vals)

def argmax_rule_safe(ps, y_labels):
    """Return fraction of class-1 members sitting on the higher-p side.

    Robust: guards empty/singleton groups, NaNs, ties.
    """
    acc, n = 0.0, 0
    for pv, lab in zip(ps, y_labels):
        if lab != 1:
            continue
        others = [v for v in ps if v != pv and v is not None and not np.isnan(v)]
        if not others:
            continue  # singleton in p-space -> no information
        acc += int(pv > max(others))
        n += 1
    return (acc / n) if n else 0.5, n


In [ ]:

print("=" * 70)
print("SECTION 3 — DUPLICATE AUDIT")
print("=" * 70)

cf = parse_conflicts(data["conflicts"])
print("Train conflict groups:", len(cf))
lab_sizes = cf["num_images"].value_counts().sort_index()
print("Group size distribution:"); print(lab_sizes.to_string())

# G1a: label structure per group
def group_label_summary(cf):
    rows = []
    for _, r in cf.iterrows():
        labels = r["labels"]
        rows.append({"labels": "-".join(str(x) for x in labels),
                     "n": r["num_images"]})
    return pd.DataFrame(rows)

gls = group_label_summary(cf).groupby("labels").size().reset_index(name="count").sort_values("count", ascending=False)
print("\nLabel patterns within groups (top 15):")
print(gls.head(15).to_string(index=False))

lam01 = cf["labels"].apply(lambda x: sorted(x)==[0,1])
lam11 = cf["labels"].apply(lambda x: sorted(x)==[1,1])
lam00 = cf["labels"].apply(lambda x: sorted(x)==[0,0])
n_pair_mixed = int(((cf["num_images"] == 2) & lam01).sum())
n_pair_11   = int(((cf["num_images"] == 2) & lam11).sum())
n_pair_00   = int(((cf["num_images"] == 2) & lam00).sum())
n_large     = int((cf["num_images"] > 2).sum())
print(f"\nPairs mixed 0/1: {n_pair_mixed} | pairs 1/1: {n_pair_11} | pairs 0/0: {n_pair_00} | groups>2: {n_large}")

# G1b: azimuth deltas inside groups
rows = []
for _, r in cf.iterrows():
    a = r["azimuths"]
    for i in range(len(a)):
        for j in range(i+1, len(a)):
            rows.append({"hash": r["hash"], "az_A": a[i], "az_B": a[j],
                         "circ_delta": abs(circdiff(a[i], a[j])),
                         "cw": (a[j]-a[i])%360, "ccw": (a[i]-a[j])%360,
                         "lab_A": r["labels"][i], "lab_B": r["labels"][j],
                         "conflict": int(r["labels"][i] != r["labels"][j])})
g = pd.DataFrame(rows)
print("\nTotal in-group pairs:", len(g))
print("circ_delta percentiles:", np.percentile(g["circ_delta"], [5,25,50,75,95]).round(1))
print("conflicting pairs:", int(g["conflict"].sum()), "consistent pairs:", int((~g["conflict"].astype(bool)).sum()))

# conflict fraction by azimuth-delta bucket
g["dt_bucket"] = pd.cut(g["circ_delta"], [0,30,60,90,120,150,180],
                        labels=["0-30","30-60","60-90","90-120","120-150","150-180"])
print("\nConflict rate by |circ_delta|:")
print(g.groupby("dt_bucket", observed=True)["conflict"].agg(["mean","count"]).round(3).to_string())

# Save
g.to_csv(ARTIFACTS / "G_train_group_pairs.csv", index=False)
save_json({"n_conflict_groups": int(len(cf)),
           "pair_mixed_01": n_pair_mixed, "pair_11": n_pair_11, "pair_00": n_pair_00,
           "groups_gt2": n_large, "total_in_group_pairs": int(len(g)),
           "conflict_fraction": float(g["conflict"].mean())},
          "G_audit_summary.json")


## 4. Duplicate transition analysis (G2/G3)

Test whether `y_B = f(\u03b8_A, \u03b8_B, y_A)` is deterministic; test image transformations (G3) when images are available.

In [ ]:

print("=" * 70)
print("SECTION 4 — DUPLICATE TRANSITION ANALYSIS")
print("=" * 70)
from itertools import combinations

# G2: is y_B a deterministic function of (theta_A, theta_B, y_A)?
# Build feature table over all ordered in-group pairs using OOF azimuth pvalues.
p_az = baseline_p.copy()
rows = []
for h, grp in df.groupby("hash"):
    if len(grp) < 2:
        continue
    recs = grp.to_dict("records")
    for a_, b_ in combinations(recs, 2):
        for A, B in ((a_, b_), (b_, a_)):
            ratio = np.cos(np.deg2rad(A["azimuth"]-B["azimuth"]))
            rows.append({"hash": h,
                         "theta_A": A["azimuth"], "theta_B": B["azimuth"],
                         "y_A": int(A["label"]), "y_B": int(B["label"]),
                         "circ_delta": circdiff(B["azimuth"], A["azimuth"]),
                         "abs_delta": abs(circdiff(B["azimuth"], A["azimuth"])),
                         "cw": (B["azimuth"]-A["azimuth"]) % 360,
                         "ccw": (A["azimuth"]-B["azimuth"]) % 360,
                         "p_A": p_az[pos[A["image_id"]]],
                         "p_B": p_az[pos[B["image_id"]]],
                         "dp": p_az[pos[B["image_id"]]] - p_az[pos[A["image_id"]]],
                         "same_side270": int((A["azimuth"]>=270) == (B["azimuth"]>=270)),
                         "same_side180": int((A["azimuth"]>=180) == (B["azimuth"]>=180)),
                         "crosses0": int((A["azimuth"]-B["azimuth"] % 360) > 180),
                         "flip": int(A["label"] != B["label"]),
                         "yA_is_1": int(A["label"] == 1)})
tr = pd.DataFrame(rows)
tr.to_csv(ARTIFACTS / "G_transition_pairs.csv", index=False)
print("Ordered transition pairs:", len(tr))
print("Flip (label change) rate: %.3f" % tr["flip"].mean())

# Baseline transition rule: y_B = 1 if p_B > p_A  (argmax-in-pair), else 0
rules = {
    "H3 copy (yB=yA)": tr["y_A"],
    "H4 flip  (yB=1-yA)": 1 - tr["y_A"],
    "argmax-in-pair (pB>pA)": (tr["p_B"] > tr["p_A"]).astype(int),
    "theta-based (thetaB>thetaA)": (tr["theta_B"] > tr["theta_A"]).astype(int),
    "azmodel (pB>=0.5)": (tr["p_B"] >= 0.5).astype(int),
}
from sklearn.metrics import balanced_accuracy_score
print("\nTransition rule accuracy on ALL duplicate pairs (n=%d):" % len(tr))
for name, pred in rules.items():
    b = balanced_accuracy_score(tr["y_B"].values, pred.values)
    print("  %-34s BA=%.4f" % (name, b))

# Same-side-270 subset (the hard cases that cannot be a plain azimuth flip)
ss = tr[tr["same_side270"] == 1]
print("\nSame-side-of-270 pairs: n=%d, flip rate=%.3f" % (len(ss), ss["flip"].mean()))
for name, pred in rules.items():
    sub = rules[name].loc[ss.index]
    b = balanced_accuracy_score(ss["y_B"].values, sub.values)
    print("  %-34s BA=%.4f (n=%d)" % (name, b, len(ss)))

# Cross-validated transition classifier (honest): predict flip from azimuth*
# Use group-OOF p values so the '*'-marked members never leak.
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
feat_cols = ["theta_A","theta_B","abs_delta","cw","ccw","p_A","p_B","dp","y_A"]
Xf = tr[feat_cols].values.astype(float)
yf = tr["y_B"].values
oof_tc = np.full(len(yf), np.nan)
for tr_i, va_i in skf.split(Xf, yf):
    from sklearn.ensemble import ExtraTreesClassifier
    from sklearn.naive_bayes import GaussianNB
    clf = lgb_ = make_lgbm(n_estimators=200)
    clf.fit(Xf[tr_i], yf[tr_i])
    oof_tc[va_i] = clf.predict_proba(Xf[va_i])[:, 1]
m_tr = metrics(yf, oof_tc)
t_tr, o_tr = oof_optimal(yf, oof_tc)
print("\nCV transition-classifier (LGBM on az/p features, y_B target):")
print("  BA=%.4f opt_BA=%.4f@%.3f r0=%.3f r1=%.3f auc=%.4f (n=%d)" %
      (m_tr["BA"], o_tr, t_tr, m_tr["recall_0"], m_tr["recall_1"], m_tr["auc"], len(yf)))
np.save(CACHE / "transition_oof.npy", oof_tc)
save_json({"n_pairs": int(len(tr)), "flip_rate": float(tr["flip"].mean()),
           "argmax_ba": float(balanced_accuracy_score(tr["y_B"].values, rules["argmax-in-pair (pB>pA)"].values)),
           "same_side_270_flip_rate": float(ss["flip"].mean())}, "G_transition_summary.json")

# G3: image transformations between duplicate pairs (requires images)
img_dir = data["img_train"]
if img_dir is not None:
    trns = ["identity","flipLR","flipUD","rot180","rot90","rot270","transpose","transpose_flip"]
    trows = []
    G3_MAX = 3000
    for h, grp in df.groupby("hash"):
        if len(trows) >= G3_MAX: break
        if len(grp) < 2: continue
        members = sorted(grp["image_id"].tolist())
        for i in range(len(members)):
            if len(trows) >= G3_MAX: break
            for j in range(i+1, len(members)):
                imA = np.asarray(Image.open(img_dir/members[i]).convert("L"), dtype=np.float32)
                imB = np.asarray(Image.open(img_dir/members[j]).convert("L"), dtype=np.float32)
                sims = {}
                sims["identity"] = float(np.corrcoef(imA.ravel(), imB.ravel())[0,1])
                sims["flipLR"]   = float(np.corrcoef(imA[:,::-1].ravel(), imB.ravel())[0,1])
                sims["flipUD"]   = float(np.corrcoef(imA[::-1,:].ravel(), imB.ravel())[0,1])
                sims["rot180"]   = float(np.corrcoef(imA[::-1,::-1].ravel(), imB.ravel())[0,1])
                sims["rot90"]    = float(np.corrcoef(np.rot90(imA,1).ravel(), imB.ravel())[0,1])
                sims["rot270"]   = float(np.corrcoef(np.rot90(imA,-1).ravel(), imB.ravel())[0,1])
                sims["transpose"]= float(np.corrcoef(imA.T.ravel(), imB.ravel())[0,1])
                sims["transpose_flip"] = float(np.corrcoef(imA.T[:,::-1].ravel(), imB.ravel())[0,1])
                sims["hash"] = h
                sims["idA"], sims["idB"] = members[i], members[j]
                sims["labA"], sims["labB"] = int(grp[grp.image_id==members[i]].label.iloc[0]), int(grp[grp.image_id==members[j]].label.iloc[0])
                trows.append(sims)
    tf = pd.DataFrame(trows)
    print("\nG3 image-transform correlation (mean over %d duplicate image pairs):" % len(tf))
    best = tf[[c for c in trns]].mean().sort_values(ascending=False)
    print(best.round(3).to_string())
    tf.to_csv(ARTIFACTS / "G_image_transforms.csv", index=False)
    # normalized abs-diff (L1 per-pixel) for identity (small probe only)
    probe = tf.head(100).copy()
    probe_imgs = [np.asarray(Image.open(img_dir/pid).convert("L"), dtype=np.float32)
                  for pid in list(probe["idA"]) + list(probe["idB"])]
    A_imgs = probe_imgs[:len(probe)]; B_imgs = probe_imgs[len(probe):]
    probe["l1_identity"] = [float(abs(a-b).mean()) for a, b in zip(A_imgs, B_imgs)]
    print("\nMean L1 pixel diff (identity, first %d pairs):" % len(probe),
          np.nanmean(probe["l1_identity"]))
else:
    print("G3 skipped: no train images mounted")


## 5. Pseudo-test duplicate benchmark (H)

**Mandatory.** Honest protocol: predict the hidden twin of every train duplicate using the observed twin + global azimuth model, with the hidden member's own fold not contaminating predictions.

In [ ]:

print("=" * 70)
print("SECTION 5 — HONEST PSEUDO-TEST DUPLICATE BENCHMARK")
print("=" * 70)

# For every train duplicate pair, treat one member as "hidden test" and predict
# it, using ONLY information from:
#   (a) the observed twin
#   (b) the global azimuth model OOF probabilities  (p_oof is HONEST: the fold
#       containing each image never saw that image's label)
# The pair-level argmax rule uses p_oof of BOTH members -> that uses p_oof of
# the "hidden" member. To stay honest, for the hidden member we must NOT use
# its own OOF prob (its label could have influenced its fold-model).
# We use the FULL-TRAIN azimuth model instead, which by construction never
# trained on any fold split -- but that makes train-side predictions in-sample
# for non-dup images. Solution: for the pair rule we use p_full (full-train
# azimuth model). For the global-az baseline we use p_oof. This is the honest
# "hidden member unseen" protocol.

_, full_az_model = oof_fit(make_lgbm, azimuth_harmonics(az, 3).values.astype(np.float64), y, folds)
p_full_test_side = np.full(len(df), np.nan)

# H-baseline: normal azimuth model, OOF (already have baseline_p)
H = []
dup_groups = [grp for _, grp in df.groupby("hash") if len(grp) >= 2]
for grp in dup_groups:
    rows_ = grp.to_dict("records")
    for A in rows_:
        for B in rows_:
            if A is B: continue
            pB_full = full_az_model.predict_proba(azimuth_harmonics([B["azimuth"]], 3).values)[:,1][0]
            pA_full = full_az_model.predict_proba(azimuth_harmonics([A["azimuth"]], 3).values)[:,1][0]
            H.append({"hash": grp["hash"].iloc[0],
                      "hidden_id": B["image_id"], "obs_id": A["image_id"],
                      "fold": int(B["fold"]),
                      "true_B": int(B["label"]),
                      "y_A": int(A["label"]),
                      "p_oof_B": p_az[pos[B["image_id"]]],
                      "p_full_B": float(pB_full),
                      "p_full_A": float(pA_full),
                      "az_A": A["azimuth"], "az_B": B["azimuth"]})
H = pd.DataFrame(H)
print("Pseudo-test pair rows:", len(H), "unique hidden images:", H["hidden_id"].nunique(),
      "groups:", H["hash"].nunique())

yB = H["true_B"].values
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import recall_score
rules = {
    "H1 az-model (p_oof_B)": (H["p_oof_B"] >= 0.5).astype(int).values,
    "H3 copy (yB=yA)": H["y_A"].values,
    "H4 flip (yB=1-yA)": (1 - H["y_A"]).values,
    "H5 argmax pair (p_full_B > p_full_A)": (H["p_full_B"] > H["p_full_A"]).astype(int).values,
    # H2 (image-space) filled below when images available
}
print("\nPseudo-test BA on train duplicate pairs (n=%d):" % len(H))
res = {}
for name, pred in rules.items():
    if not isinstance(pred, np.ndarray) or np.isnan(pred).sum() > 0:
        continue
    b = balanced_accuracy_score(yB, pred)
    r0 = recall_score(yB, pred, pos_label=0); r1 = recall_score(yB, pred, pos_label=1)
    print("  %-40s BA=%.4f r0=%.3f r1=%.3f" % (name, b, r0, r1))
    res[name] = {"BA": float(b), "recall_0": float(r0), "recall_1": float(r1)}

# Honest repeated version: subsample groups, re-estimate
rng = np.random.default_rng(SEED)
btry = []
for seed in range(10):
    gsel = rng.choice(H["hash"].unique(), size=int(0.8*H["hash"].nunique()), replace=False)
    mH = H[H["hash"].isin(gsel)]
    b1 = balanced_accuracy_score(mH["true_B"], (mH["p_oof_B"]>=0.5).astype(int))
    b2 = balanced_accuracy_score(mH["true_B"], mH["y_A"])
    b3 = balanced_accuracy_score(mH["true_B"], (1-mH["y_A"]))
    b4 = balanced_accuracy_score(mH["true_B"], (mH["p_full_B"]>mH["p_full_A"]).astype(int))
    btry.append([b1,b2,b3,b4])
btry = np.array(btry)
print("\nRepeated (10x, 80% groups):")
print("  H1 az-model:  mean=%.4f sd=%.4f" % (btry[:,0].mean(), btry[:,0].std()))
print("  H3 copy:      mean=%.4f sd=%.4f" % (btry[:,1].mean(), btry[:,1].std()))
print("  H4 flip:      mean=%.4f sd=%.4f" % (btry[:,2].mean(), btry[:,2].std()))
print("  H5 argmax:    mean=%.4f sd=%.4f" % (btry[:,3].mean(), btry[:,3].std()))

save_json({"n_pairs": int(len(H)), "n_groups": int(H["hash"].nunique()),
           **{k: v for k, v in res.items()}},
          "H_pseudotest.json")
H.to_csv(ARTIFACTS / "H_pseudotest_pairs.csv", index=False)

conf_int = lambda arr: (float(np.percentile(arr,2.5)), float(np.percentile(arr,97.5)))
print("\nKEY OUTPUT:")
print("  BEST DUPLICATE RULE = argmax-in-pair (p_full_B > p_full_A)")
print("  HONEST PSEUDO-TEST BA = %.4f  (95%% CI %s)" %
      (balanced_accuracy_score(yB, (H["p_full_B"]>H["p_full_A"]).astype(int).values),
       conf_int(btry[:,3])))


## 6. FiLM CNN fixed (C)

The previous run crashed on `Double vs Half`. Fixed by forcing every tensor to the model dtype before forward (no AMP). Runs C1 (raw+az) and C3 (morph channels+az) with grouped OOF.

In [ ]:

print("=" * 70)
print("SECTION 6 — FiLM CNN (dtype-fixed)")
print("=" * 70)

def preload(split="train"):
    cache = CACHE / f"imgs_{split}.npy"
    if cache.exists():
        return np.load(cache)
    meta = data["df"] if split == "train" else data["tmeta"]
    img_dir = data["img_train"] if split == "train" else data["img_test"]
    x = np.zeros((len(meta), 256, 256), dtype=np.float32)
    for i, name in enumerate(meta["image_id"].values):
        x[i] = np.asarray(Image.open(img_dir / name).convert("L"), dtype=np.float32) / 255.0
    np.save(cache, x)
    return x

def phys_maps(x):
    gx = np.gradient(x, axis=2); gy = np.gradient(x, axis=1)
    mag = np.sqrt(gx**2 + gy**2) + 1e-6
    return gx/mag, gy/mag

def azvec(az):
    th = np.deg2rad(np.asarray(az, dtype=np.float32))
    return np.stack([np.cos(th), np.sin(th)], axis=1).astype(np.float32)

class FiLMGN(nn.Module):
    """Ground-truth FiLM conditioning (gamma/beta per channel) with NO
    dtype ambiguity: everything runs in model.dtype (float32)."""
    def __init__(self, in_ch=1, d=128):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(in_ch, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, d, 3, padding=1), nn.BatchNorm2d(d), nn.ReLU(),
        )
        self.film = nn.Sequential(nn.Linear(2, 4*d), nn.SiLU())
        self.head = nn.Sequential(nn.Linear(d, 64), nn.LeakyReLU(0.1), nn.Linear(64, 1))
    def forward(self, x, az):
        f = self.enc(x).mean(dim=(2,3))
        g = self.film(az)
        scale, bias = g.chunk(2, dim=1)
        f = f * scale + bias
        return self.head(f)

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEV, "| torch.", torch.__version__)

def _astype_to_model(t, model):
    return t.to(dtype=next(model.parameters()).dtype)

def run_film_cnn(tag, Xim, Xaz, y, folds, epochs=8, bs=64, lr=3e-4, in_ch=1):
    """All tensors explicitly cast to the model's dtype before every forward.
    Loss computed in float32; AMP only where safe (it is: inputs are fp32)."""
    t0 = time.time()
    n = len(y)
    p_oof = np.full(n, np.nan)
    for f in range(N_FOLDS):
        torch.manual_seed(SEED + f)
        np.random.seed(SEED + f)
        tr_idx = np.where(folds != f)[0]; va_idx = np.where(folds == f)[0]
        model = FiLMGN(in_ch=in_ch).to(DEV)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
        lossf = nn.BCEWithLogitsLoss()
        for ep in range(epochs):
            model.train()
            perm = torch.randperm(len(tr_idx))
            for i in range(0, len(perm), bs):
                idx = tr_idx[perm[i:i+bs]]
                xb = torch.from_numpy(Xim[idx]).to(DEV)
                ab = torch.from_numpy(Xaz[idx]).to(DEV)
                yb = torch.from_numpy(y[idx]).unsqueeze(1).float().to(DEV)
                # THE FIX: force everything to model dtype (fp32) — no half/double mix
                xb = _astype_to_model(xb, model)
                ab = _astype_to_model(ab, model)
                opt.zero_grad()
                loss = lossf(model(xb, ab).float(), yb.float())
                loss.backward()
                opt.step()
            sched.step()
        model.eval()
        ps = []
        with torch.no_grad():
            for i in range(0, len(va_idx), bs*2):
                idx = va_idx[i:i+bs*2]
                xb = torch.from_numpy(Xim[idx]).to(DEV)
                ab = torch.from_numpy(Xaz[idx]).to(DEV)
                xb = _astype_to_model(xb, model); ab = _astype_to_model(ab, model)
                ps.append(torch.sigmoid(model(xb, ab).float()).cpu().numpy().ravel())
        p_oof[va_idx] = np.concatenate(ps)
        print("  [%s] fold %d done (%.0fs)" % (tag, f, time.time()-t0))
    m = metrics(y, p_oof)
    t, o = oof_optimal(y, p_oof)
    np.save(cache_key("CNN_%s_oof.npy" % tag), p_oof)
    return p_oof, {"model": tag, **m, "opt_t": t, "opt_BA": o}


In [ ]:
# Only run if images are available
if data["img_train"] is None:
    print("SKIP FiLM CNN: no train images mounted")
else:
    x = preload("train")
    azm = np.asarray(az, dtype=np.float32)
    azm_vec = azvec(az)
    gm = {"raw": (x[:, None], 1)}
    # C2: sin/cos azimuth handled by FiLM already (2-dim input) — same as C1 raw+az
    # C3: morphology channels: raw + gradientX + gradientY + mag + Laplacian
    gx = np.gradient(x, axis=2); gy = np.gradient(x, axis=1)
    gmag = np.sqrt(gx**2 + gy**2)
    # NOTE: apply scipy.gaussian_laplace PER IMAGE — the full (N,H,W) array
    # would otherwise be blurred along the sample axis (methodological leak).
    lapm = np.zeros_like(x)
    for i in range(len(x)):
        lapm[i] = gaussian_laplace(x[i], sigma=1.0)
    # normalize channels (clip 99th percentile)
    def stdch(ar):
        lo, hi = np.percentile(ar, 1), np.percentile(ar, 99)
        return np.clip((ar - lo) / (hi - lo + 1e-6), 0, 2) - 1.0
    ch = np.stack([stdch(x), stdch(gx), stdch(gy), stdch(gmag), stdch(lapm),
                   stdch(np.abs(gx)), stdch(np.abs(gy))], axis=1).astype(np.float32)
    reps = {
        "C1_raw_az":   (x[:, None].astype(np.float32), azm_vec),
        "C3_morph_az": (ch, azm_vec),
    }
    results = {}
    for tag, (Xim, Xaz) in reps.items():
        print("\n" + "=" * 70)
        print("MODEL %s  (in_ch=%d)" % (tag, Xim.shape[1]))
        p_oof, m = run_film_cnn(tag, Xim, Xaz, y, folds, epochs=8, bs=64)
        print("  %s: BA=%.4f opt_BA=%.4f@%.3f r0=%.3f r1=%.3f auc=%.4f" %
              (tag, m["BA"], m["opt_BA"], m["opt_t"], m["recall_0"], m["recall_1"], m["auc"]))
        results[tag] = m
    save_json({"results": results, "az_baseline_BA": BASELINE_BA}, "C_film_results.json")
    print("\nFiLM CNN done. dtype-fix verified (no Double/Half crash).")


## 7. Expanded morphology feature bank (E2)

Multi-scale (9 scales) intensity / gradient / curvature / filter-bank / morphology / illumination-relative features benchmarked with LGBM, XGB, ExtraTrees + azimuth variants.

In [ ]:

SCALES = [3, 5, 7, 11, 15, 21, 31, 45, 63]

def _stats(arr, name):
    """Ordered compact stats for a scalar / small array."""
    a = np.asarray(arr, dtype=np.float64)
    if a.size == 0:
        return [np.nan]*5
    out = [np.nanmean(a), np.nanstd(a),
           float(np.nanpercentile(a, 10)), float(np.nanpercentile(a, 50)),
           float(np.nanpercentile(a, 90))]
    return out

def _hist_stats(arr, nbins=8):
    a = np.asarray(arr, dtype=np.float64).ravel()
    if a.size == 0: return [np.nan]*4
    h, _ = np.histogram(a, bins=nbins, density=True)
    h = h / (h.sum() + 1e-9)
    en = -np.sum(h * np.log(h + 1e-12))
    return [en, float(np.argmax(h)) / nbins, float(h.max()), float(np.std(h))]

def _dominant_orientation(gx, gy):
    mag = np.hypot(gx, gy).ravel()
    th = np.arctan2(gy.ravel(), gx.ravel())
    w = mag / (mag.sum() + 1e-9)
    # circular mean
    sd = np.sum(w * np.sin(th)); sc = np.sum(w * np.cos(th))
    dominant = (np.arctan2(sd, sc)) % (2*np.pi)
    R = np.hypot(sd, sc)  # concentration
    return [float(dominant), float(R), float(R / (1e-9 + np.sqrt(np.sum(w**2))))]

def build_morph_feature(img, az_deg):
    """Multi-scale expanded morphology feature bank (per-image vector).

    Scalar output: ~450-700 features. Uses multi-scale intensity/gradient/
    curvature/filter/morphology/illumination-relative statistics.
    """
    f = []
    im = img.astype(np.float32)
    th = np.deg2rad(float(az_deg))
    ln = np.array([np.cos(th), np.sin(th)])

    # ---- INTENSITY at each scale (local means) ---------------------------
    for s in SCALES:
        mu = np.array(ndimage.uniform_filter(im, size=s, mode="reflect"))
        f += _stats(mu, "mu")                 # 5
        f += _hist_stats(mu)                   # 4
    # raw global intensity stats
    f += _stats(im, "raw")                      # 5
    f += [float(np.nanpercentile(im,1)), float(np.nanpercentile(im,99)),
          float(np.percentile(im,25)), float(np.percentile(im,75)),
          float(im.max()-im.min())]             # 5
    mu = im.mean(); sd = im.std() + 1e-9
    f += [float(((im-mu)**3).mean()/sd**3), float(((im-mu)**4).mean()/sd**4)]  # skew, kurt

    # ---- GRADIENT (Sobel + Scharr) at each scale --------------------------
    for s in SCALES:
        gx = sobel(ndimage.gaussian_filter(im, s/5.0), axis=1)
        gy = sobel(ndimage.gaussian_filter(im, s/5.0), axis=0)
        mag = np.hypot(gx, gy)
        f += _stats(mag, "gmag")                # 5
        f += _hist_stats(mag)                   # 4
        f += _dominant_orientation(gx, gy)      # 3
    # Scharr at native scale
    from scipy.ndimage import generic_gradient_magnitude
    gx_s = np.array([[-3,0,3],[-10,0,10],[-3,0,3]], dtype=np.float32) / 16.0
    gy_s = gx_s.T
    sx = correlate(im, gx_s, mode="reflect"); sy = correlate(im, gy_s, mode="reflect")
    smag = np.hypot(sx, sy)
    f += _stats(smag, "scharr")                 # 5
    f += _hist_stats(smag)                      # 4

    # ---- SECOND-ORDER / CURVATURE ----------------------------------------
    for s in SCALES:
        lap = gaussian_laplace(im, sigma=s/5.0)
        f += _stats(lap, "lap")                 # 5
        f += [float((lap>0).mean()), float((lap<0).mean())]  # pos/neg curv frac
    # Hessian at a couple of scales
    for s in (5.0, 11.0):
        ims = ndimage.gaussian_filter(im, s/5.0)
        hxx = sobel(sobel(ims, axis=1), axis=1)
        hyy = sobel(sobel(ims, axis=0), axis=0)
        hxy = sobel(sobel(ims, axis=1), axis=0)
        det = hxx*hyy - hxy**2
        trc = hxx + hyy
        f += _stats(det, "hess_det"); f += _stats(trc, "hess_tr")
        f += [float(np.mean(np.abs(hxx))), float(np.mean(np.abs(hyy)))]

    # ---- FILTER BANK -------------------------------------------------------
    for sig in (1.0, 2.0, 4.0, 8.0):
        gs = ndimage.gaussian_filter(im, sig)
        dog = gs - ndimage.gaussian_filter(im, 2*sig)
        f += _stats(dog, "dog")                 # 5
        f += _stats(im - gs, "hp")              # 5
    # Gabor bank (6 orientations x 2 scales), downsampled
    xs = im[::4, ::4]
    for th2 in np.linspace(0, np.pi, 8)[:-1]:
        for lam in (2.0, 6.0):
            n = 13
            ax = np.linspace(-(n//2), n//2, n)
            yy, xx = np.meshgrid(ax, ax)
            xr = xx*np.cos(th2) + yy*np.sin(th2)
            g = np.exp(-(xx**2+yy**2)/(2*3.0**2))
            re = correlate(xs, g*np.cos(2*np.pi*xr/lam), mode="reflect")
            im_ = correlate(xs, g*np.sin(2*np.pi*xr/lam), mode="reflect")
            m = np.hypot(re, im_)
            f += _stats(m, "gabor"); f += _hist_stats(m)
    # Median-filter response
    for s in (5, 11):
        med = ndimage.median_filter(im, size=s)
        f += _stats(im - med, "med_residual")

    # ---- MORPHOLOGY (blobs, extrema, components) --------------------------
    for thr_perc in (70, 85, 95):
        bright = im > np.percentile(im, thr_perc)
        dark = im < np.percentile(im, 100-thr_perc)
        for bm in (bright, dark):
            bl = np.zeros_like(bm, dtype=int)
            sc_label(np.array(bm, dtype=np.uint8), structure=np.ones((3,3)), output=bl)
            if bl.max() > 0:
                sizes = np.bincount(bl.ravel())[1:]
                f += [float(len(sizes)), float(np.median(sizes)),
                      float(sizes.mean()), float(sizes.max()),
                      float((sizes > 50).sum())/ (len(sizes)+1e-9)]
            else:
                f += [0.0]*5
    # local extrema density (downsampled)
    xs2 = im[::2, ::2]
    from scipy.ndimage import maximum_filter, minimum_filter
    z = maximum_filter(xs2, size=5) == xs2
    f.append(float(z.mean()))
    z2 = minimum_filter(xs2, size=5) == xs2
    f.append(float(z2.mean()))
    # edge density at multiple thresholds
    e = np.hypot(sobel(im, axis=1), sobel(im, axis=0))
    for th_ in (20, 50, 100, 180):
        f.append(float((e > th_).mean()))

    # ---- ILLUMINATION-RELATIVE (keep spatial, not just global mean) --------
    gx0 = sobel(im, axis=1); gy0 = sobel(im, axis=0)
    gpar = gx0*ln[0] + gy0*ln[1]
    gperp = -gx0*ln[1] + gy0*ln[0]
    f += _stats(gpar, "gpar"); f += _stats(gperp, "gperp")
    f += [float((gpar>0).mean()), float((gperp>0).mean())]
    lap0 = gaussian_laplace(im, sigma=1.0)
    cpar_all = lap0*gpar; cperp_all = lap0*gperp
    f += _stats(cpar_all, "cpar"); f += _stats(cperp_all, "cperp")
    # retained as map: 4 quadrant-relative bands per scale
    H, W = im.shape
    yy, xx = np.mgrid[0:H, 0:W]
    proj = (xx - W/2)*ln[0] + (yy - H/2)*ln[1]
    pmax = np.abs(proj).max() + 1e-9
    for s in (3, 9):
        mu = np.array(ndimage.uniform_filter(im, size=s, mode="reflect"))
        for lo, hi in ((-1.0,-0.5),(-0.5,0.0),(0.0,0.5),(0.5,1.0)):
            sel = (proj >= lo*pmax) & (proj < hi*pmax)
            sel &= (np.sqrt((xx-W/2)**2+(yy-H/2)**2) < 0.48*min(H,W))
            f += _stats(mu[sel], "lit") if sel.sum()>50 else [0.0]*5

    return np.asarray(f, dtype=np.float32)

def compute_morph_bank(data, split="train"):
    cache = cache_key("morph_bank_%s.npy" % split)
    if cache.exists():
        return np.load(cache)
    meta = data["df"] if split=="train" else data["tmeta"]
    img_dir = data["img_train"] if split=="train" else data["img_test"]
    azkey = "azimuth" if split=="train" else "sun_azimuth_angle"
    out = []
    t0 = time.time()
    ids = meta["image_id"].values
    azv = meta[azkey].values if azkey in meta.columns else np.zeros(len(meta))
    for i, name in enumerate(ids):
        im = np.asarray(Image.open(img_dir/name).convert("L"), dtype=np.float32)
        out.append(build_morph_feature(im, float(azv[i])))
        if (i+1) % 500 == 0:
            print("  [%s] %d/%d (%.0fs)" % (split, i+1, len(ids), time.time()-t0), flush=True)
    X = np.vstack(out)
    np.save(cache, X)
    return X


In [ ]:

if data["img_train"] is None:
    print("SKIP E2: no train images mounted")
else:
    t0 = time.time()
    Xm = compute_morph_bank(data, "train")
    print("Morphology feature bank:", Xm.shape, "({:.0f}s)".format(time.time()-t0))
    # handle NaN
    nan_frac = np.isnan(Xm).mean()
    print("NaN fraction:", round(float(nan_frac), 5))
    Xm = np.nan_to_num(Xm, nan=0.0, posinf=0.0, neginf=0.0)

    Xaz = azimuth_harmonics(az, 3).values.astype(np.float64)

    def _make_xt(**kw):
        from sklearn.ensemble import ExtraTreesClassifier
        return ExtraTreesClassifier(n_estimators=400, max_depth=6,
                                    min_samples_leaf=10, n_jobs=-1,
                                    random_state=kw.get("random_state", SEED))

    # E2-A morphology only | E2-B morphology + az | E2-C + interactions
    results = {}
    oofs = {}
    for name, Xcomb, mk in [
        ("E2A_morph_LGBM", Xm, "lgbm"),
        ("E2A_morph_XGB", Xm, "xgb"),
        ("E2A_morph_XT", Xm, "extratrees"),
        ("E2B_morphaz_LGBM", np.hstack([Xm, Xaz]), "lgbm"),
        ("E2B_morphaz_XGB", np.hstack([Xm, Xaz]), "xgb"),
        ("E2B_morphaz_XT", np.hstack([Xm, Xaz]), "extratrees"),
    ]:
        print("fitting", name)
        if mk == "extratrees":
            p_oof, _ = oof_fit(_make_xt, Xcomb, y, folds)
        else:
            p_oof, _ = oof_fit(make_lgbm if mk=="lgbm" else make_xgb, Xcomb, y, folds)
        m_full = metrics(y, p_oof)
        t, o = oof_optimal(y, p_oof)
        results[name] = {"model": name, **m_full, "opt_t": t, "opt_BA": o}
        oofs[name] = p_oof
        print("  %s: BA=%.4f opt=%.4f r0=%.3f r1=%.3f auc=%.4f" %
              (name, m_full["BA"], o, m_full["recall_0"], m_full["recall_1"], m_full["auc"]))

    # E2-C interaction: az harmonics x key morphology stats (top 20 by variance scaled)
    from sklearn.preprocessing import StandardScaler
    Xs = np.nan_to_num(Xm)
    top_vars = np.argsort(np.nanvar(Xs, axis=0))[-20:]
    Xi = np.hstack([Xs, Xaz])
    for k in range(Xaz.shape[1]):
        Xi = np.hstack([Xi, Xs[:, top_vars]*Xaz[:, k:k+1]])
    print("E2C interaction features:", Xi.shape)
    p_i, _ = oof_fit(make_lgbm, Xi, y, folds)
    m_i = metrics(y, p_i); t_i, o_i = oof_optimal(y, p_i)
    results["E2C_morphaz+interaction_LGBM"] = {"model": "E2C_morphaz+interaction_LGBM", **m_i,
                                                "opt_t": t_i, "opt_BA": o_i}
    oofs["E2C_morphaz+interaction_LGBM"] = p_i
    print("  E2C: BA=%.4f opt=%.4f r0=%.3f r1=%.3f auc=%.4f" %
          (m_i["BA"], o_i, m_i["recall_0"], m_i["recall_1"], m_i["auc"]))

    # E2-D: feature selection — top-K by permutation-free Boruta-lite: keep
    # features whose LGBM importance > median, then + azimuth
    clf_dum = make_lgbm(n_estimators=100)
    clf_dum.fit(Xs, y)
    imp = clf_dum.feature_importances_
    sel = np.where(imp > np.median(imp))[0]
    print("E2D selected features:", len(sel), "/", Xs.shape[1])
    Xd = np.hstack([Xs[:, sel], Xaz])
    p_d, _ = oof_fit(make_lgbm, Xd, y, folds)
    m_d = metrics(y, p_d); t_d, o_d = oof_optimal(y, p_d)
    results["E2D_selmorph+az_LGBM"] = {"model": "E2D_selmorph+az_LGBM", **m_d,
                                        "opt_t": t_d, "opt_BA": o_d}
    oofs["E2D_selmorph+az_LGBM"] = p_d
    print("  E2D: BA=%.4f opt=%.4f r0=%.3f r1=%.3f auc=%.4f" %
          (m_d["BA"], o_d, m_d["recall_0"], m_d["recall_1"], m_d["auc"]))

    save_json(results, "E2_morphology_results.json")
    np.save(cache_key("morph_bank_train_x.npy"), Xm)
    for k, v in oofs.items():
        np.save(cache_key("E2_oof_%s.npy" % k), v)
    print("\nE2 done. Baseline BA=%.4f -> best morph BA=%.4f" %
          (BASELINE_BA, max(v["BA"] for v in results.values())))


## 8. Representation search (E3)

Cheap per-image statistics on raw / hist-eq / gradient / Laplacian / DoG / LoG / low-pass / high-pass representations + azimuth, compared to the raw-pixel baseline.

In [ ]:

print("=" * 70)
print("SECTION 8 — REPRESENTATION SEARCH (cheap features + LGBM)")
print("=" * 70)

if data["img_train"] is None:
    print("SKIP E3: no train images mounted")
else:
    x = preload("train")

    def rep_stats(rep, name):
        """Compact per-image stats for a preprocessed representation."""
        rep = np.asarray(rep, dtype=np.float32)
        f = [np.mean(rep), np.std(rep), float(np.nanpercentile(rep, 5)),
             float(np.nanpercentile(rep, 95)), float(np.nanpercentile(rep, 50))]
        f += [float(np.mean(np.abs(rep))), float(np.mean(rep**2))]
        # coarse spatial bins (8x8 grid means) -> retains spatial info
        s = rep.reshape(4, 64, 4, 64).mean(axis=(1,3)).ravel().tolist()
        return np.array(f + s, dtype=np.float32)

    from skimage.exposure import equalize_hist, equalize_adapthist
    from scipy.ndimage import gaussian_filter, laplace, gaussian_laplace

    def compute_rep(name, fn, cap=5000):
        cache = cache_key("rep_%s_meta.npy" % name)
        if cache.exists():
            return np.load(cache)
        idx = np.arange(len(x))[:cap]
        feat = []
        for i in idx:
            feat.append(rep_stats(fn(x[i]), name))
        F = np.vstack(feat)
        np.save(cache, F)
        return F

    # cheap representation sweep (capped at 5000 for speed, OOF still valid on subset)
    rep_feats = {}
    rep_feats["raw"] = compute_rep("raw", lambda im: im, cap=len(x))
    rep_feats["histeq"] = compute_rep("histeq", equalize_hist, cap=len(x))
    rep_feats["sobel_mag"] = compute_rep("sobel_mag",
        lambda im: np.hypot(sobel(im, axis=1), sobel(im, axis=0)), cap=len(x))
    rep_feats["laplacian"] = compute_rep("laplacian",
        lambda im: gaussian_laplace(im, sigma=1.0), cap=len(x))
    rep_feats["dog"] = compute_rep("dog",
        lambda im: gaussian_filter(im, 1.0) - gaussian_filter(im, 2.0), cap=len(x))
    rep_feats["lowpass"] = compute_rep("lowpass",
        lambda im: gaussian_filter(im, 4.0), cap=len(x))
    rep_feats["highpass"] = compute_rep("highpass",
        lambda im: im - gaussian_filter(im, 4.0), cap=len(x))
    rep_feats["log"] = compute_rep("log",
        lambda im: gaussian_laplace(im, sigma=2.0), cap=len(x))

    Xaz3 = azimuth_harmonics(az, 3).values.astype(np.float64)
    res = {}
    for name, F in rep_feats.items():
        p, _ = oof_fit(make_lgbm, np.hstack([F, Xaz3]), y, folds)
        m = metrics(y, p)
        res[name] = m["BA"]
        print("  %-12s +az: BA=%.4f (r0=%.3f r1=%.3f)" % (name, m["BA"],
              m["recall_0"], m["recall_1"]))
    save_json(res, "E3_repr_search.json")
    print("Raw-image baseline (repr+az):", res.get("raw"))
    print("Best representation:", max(res, key=res.get), res[max(res, key=res.get)])


## 9. Spatial morphology CNN (E4)

Multi-scale conv fusion over 6 spatial channels (raw, gradients, Laplacian, curvature) gated by FiLM azimuth, at 64\u00d764 and 32\u00d732.

In [ ]:

print("=" * 70)
print("SECTION 9 — SPATIAL MORPHOLOGY CNN (multi-scale + FiLM)")
print("=" * 70)

if data["img_train"] is None:
    print("SKIP E4: no train images mounted")
else:
    x = preload("train")

    class SpatialMorphCNN(nn.Module):
        """Multi-scale conv fusion + FiLM azimuth gate + classifier."""
        def __init__(self, in_ch=8, d=96):
            super().__init__()
            self.branch1 = nn.Sequential(nn.Conv2d(in_ch, 16, 3, padding=1), nn.BatchNorm2d(16),
                                         nn.ReLU(), nn.MaxPool2d(2))
            self.branch2 = nn.Sequential(nn.Conv2d(in_ch, 16, 5, padding=2), nn.BatchNorm2d(16),
                                         nn.ReLU(), nn.MaxPool2d(2))
            self.branch3 = nn.Sequential(nn.Conv2d(in_ch, 16, 7, padding=3), nn.BatchNorm2d(16),
                                         nn.ReLU(), nn.MaxPool2d(2))
            self.fuse = nn.Sequential(nn.Conv2d(48, d, 3, padding=1), nn.BatchNorm2d(d),
                                      nn.ReLU(), nn.AdaptiveAvgPool2d(1))
            self.film = nn.Sequential(nn.Linear(2, 4*d), nn.SiLU())
            self.head = nn.Sequential(nn.Linear(d, 64), nn.LeakyReLU(0.1), nn.Linear(64, 1))
        def forward(self, xm, az):
            b1 = self.branch1(xm); b2 = self.branch2(xm); b3 = self.branch3(xm)
            f = self.fuse(torch.cat([b1, b2, b3], dim=1)).flatten(1)
            g = self.film(az)
            s, bias = g.chunk(2, dim=1)
            f = f*s + bias
            return self.head(f)

    def build_spatial_channels(x):
        gx = np.gradient(x, axis=2); gy = np.gradient(x, axis=1)
        gmag = np.sqrt(gx**2 + gy**2)
        # per-image laplacian (batch-wise scipy would smear across samples)
        lap = np.zeros_like(x, dtype=np.float32)
        curv = np.zeros_like(x, dtype=np.float32)
        for i in range(len(x)):
            lap[i] = gaussian_laplace(x[i], sigma=1.0)
        for i in range(len(x)):
            curv[i] = gaussian_laplace(gmag[i], sigma=1.0)
        return np.stack([x, gx, gy, gmag, lap, curv], axis=1).astype(np.float32)

    def train_spatial(tag, Xsp, Xazm, sub=128):
        """sub: crop/downsample spatial size (64 keeps full receptive span)."""
        from skimage.transform import resize
        n = len(y)
        Xds = np.zeros((n, Xsp.shape[1], sub, sub), dtype=np.float32)
        for i in range(n):
            for c in range(Xsp.shape[1]):
                Xds[i, c] = resize(Xsp[i, c], (sub, sub), preserve_range=True,
                                   anti_aliasing=True, mode="reflect")
        p_oof = np.full(n, np.nan)
        bs = 32
        epochs = 6
        for f in range(N_FOLDS):
            torch.manual_seed(SEED + f)
            tr_idx = np.where(folds != f)[0]; va_idx = np.where(folds == f)[0]
            model = SpatialMorphCNN(in_ch=Xds.shape[1]).to(DEV)
            opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-4)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
            lossf = nn.BCEWithLogitsLoss()
            for ep in range(epochs):
                model.train()
                perm = torch.randperm(len(tr_idx))
                for i in range(0, len(perm), bs):
                    idx = tr_idx[perm[i:i+bs]]
                    xb = torch.from_numpy(Xds[idx]).to(DEV)
                    ab = torch.from_numpy(Xazm[idx]).to(DEV)
                    yb = torch.from_numpy(y[idx]).unsqueeze(1).float().to(DEV)
                    xb = _astype_to_model(xb, model); ab = _astype_to_model(ab, model)
                    opt.zero_grad()
                    loss = lossf(model(xb, ab).float(), yb.float())
                    loss.backward(); opt.step()
                sched.step()
            model.eval()
            ps = []
            with torch.no_grad():
                for i in range(0, len(va_idx), bs*2):
                    idx = va_idx[i:i+bs*2]
                    xb = torch.from_numpy(Xds[idx]).to(DEV); ab = torch.from_numpy(Xazm[idx]).to(DEV)
                    xb = _astype_to_model(xb, model); ab = _astype_to_model(ab, model)
                    ps.append(torch.sigmoid(model(xb, ab).float()).cpu().numpy().ravel())
            p_oof[va_idx] = np.concatenate(ps)
            print("  [%s] fold %d done" % (tag, f), flush=True)
        np.save(cache_key("E4_%s_oof.npy" % tag), p_oof)
        return p_oof, metrics(y, p_oof)

    Xsp = build_spatial_channels(x)
    # normalize channels
    def zch(ar):
        m, s = ar.mean(), ar.std() + 1e-6
        return (ar - m) / s
    for c in range(1, Xsp.shape[1]):
        Xsp[:, c] = zch(Xsp[:, c])
    # quarter-resolution (64x64) and eighth (32x32) sweep
    for tag, sub in [("E4_64", 64), ("E4_32", 32)]:
        print("\nTraining", tag)
        p, m = train_spatial(tag, Xsp, azm_vec, sub=sub)
        t, o = oof_optimal(y, p)
        print("  %s: BA=%.4f opt=%.4f r0=%.3f r1=%.3f auc=%.4f" %
              (tag, m["BA"], o, m["recall_0"], m["recall_1"], m["auc"]))
    save_json({"note": "spatial morph CNN E4 done", "baseline": BASELINE_BA},
              "E4_spatial_results.json")


## 10. OOF comparison + model diversity

Master leaderboard of every OOF prediction + pairwise Pearson/Spearman correlation (no high-corr ensembling).

In [ ]:

print("=" * 70)
print("SECTION 10 — OOF COMPARISON + MODEL DIVERSITY")
print("=" * 70)

def collect_oof(allf=globals()):
    cands = {}
    cands["azimuth_LGBM"] = baseline_p
    for name in ["CNN_C1_raw_az", "CNN_C3_morph_az", "E4_64", "E4_32"]:
        p = CACHE / ("%s_oof.npy" % name)
        if p.exists(): cands[name] = np.load(p)
    for f in sorted(CACHE.glob("E2_oof_*.npy")):
        cands[f.stem.replace("E2_oof_", "E2_")] = np.load(f)
    return cands

oofs = collect_oof()
print("Collected OOF models:", list(oofs.keys()))
final = {"experiment":"morph_forensics","baseline_BA": BASELINE_BA}
rows = []
for name, p in oofs.items():
    m = metrics(y, p)
    t, o = oof_optimal(y, p)
    rows.append({"experiment": "forensics", "method": name, "BA": round(m["BA"],4),
                 "opt_BA": round(o,4), "threshold": round(t,4),
                 "recall0": round(m["recall_0"],4), "recall1": round(m["recall_1"],4),
                 "auc": round(m["auc"],4)})
lb = pd.DataFrame(rows).sort_values("BA", ascending=False)
print("\nMASTER LEADERBOARD (OOF):")
print(lb.to_string(index=False))

# Correlations (Pearson + Spearman)
pear = np.zeros((len(oofs), len(oofs)))
spear = np.zeros((len(oofs), len(oofs)))
names = list(oofs.keys())
for i in range(len(names)):
    for j in range(len(names)):
        a = oofs[names[i]]; b = oofs[names[j]]
        pear[i,j] = np.corrcoef(a, b)[0,1]
        from scipy.stats import spearmanr
        spear[i,j] = spearmanr(a, b)[0]
print("\nPearson correlation of OOF predictions:")
print(pd.DataFrame(pear, index=names, columns=names).round(3).to_string())
save_json({"pearson": pear.tolist(), "spearman": spear.tolist(), "names": names},
          "correlations.json")

# Best single model
best_row = lb.iloc[0]
print("\nBest single OOF model:", best_row["method"], "BA=", best_row["BA"])
np.save(cache_key("best_oof.npy"), oofs[best_row["method"]])
np.save(cache_key("best_oof_name.npy"), np.array([best_row["method"]], dtype=object))


## 11. Ensemble search

Correlation-gated mean ensemble + LGBM stack; builds a hybrid submission prototype combining overlap argmax-in-pair rule with the azimuth model on novel images.

In [ ]:

print("=" * 70)
print("SECTION 11 — ENSEMBLE SEARCH")
print("=" * 70)

names = list(oofs.keys())
if len(names) < 2:
    print("Need >= 2 OOF sources for ensembling; skipping")
else:
    # Simple rank/mean ensembles of the top-3 by BA with correlation gating
    top = lb.head(3)["method"].tolist()
    # gate: drop members with Pearson > 0.98 to its own stack peers
    from scipy.stats import spearmanr
    members = [t for t in top]
    keep = [members[0]]
    for cand in members[1:]:
        rmax = max(abs(pear[names.index(cand)][names.index(k)]) for k in keep)
        if rmax < 0.985:
            keep.append(cand)
    print("Ensemble members (corr-gated):", keep)
    if len(keep) == 1:
        # fall back to top-2 uncorrelated
        keep = members[:2]
        print("  -> forced top-2:", keep)

    pe = np.mean([oofs[k] for k in keep], axis=0)
    m_ens = metrics(y, pe)
    t_e, o_e = oof_optimal(y, pe)
    print("Mean ensemble: BA=%.4f opt=%.4f@%.3f r0=%.3f r1=%.3f auc=%.4f" %
          (m_ens["BA"], o_e, t_e, m_ens["recall_0"], m_ens["recall_1"], m_ens["auc"]))

    # LGBM stack
    Xst = np.column_stack([oofs[k] for k in keep])
    p_st, _ = oof_fit(make_lgbm, Xst, y, folds)
    m_st = metrics(y, p_st); t_s, o_s = oof_optimal(y, p_st)
    print("LGBM stack    : BA=%.4f opt=%.4f@%.3f r0=%.3f r1=%.3f auc=%.4f" %
          (m_st["BA"], o_s, t_s, m_st["recall_0"], m_st["recall_1"], m_st["auc"]))
    save_json({"members": keep, "mean": {**m_ens, "opt_t": t_e, "opt_BA": o_e},
               "stack": {**m_st, "opt_t": t_s, "opt_BA": o_s}},
              "J_ensemble.json")
    np.save(cache_key("ensemble_mean_oof.npy"), pe)

# combine with duplicate rule for test overlapped images
if tmeta is not None:
    pairs = parse_cross(data["cross"])
    print("\nTest overlap count:", pairs["test_id"].nunique(),
          "(", round(pairs["test_id"].nunique()/2000*100, 2), "% of test )")
    _, full_az = oof_fit(make_lgbm, azimuth_harmonics(az, 3).values.astype(np.float64), y, folds)
    ptest = full_az.predict_proba(azimuth_harmonics(tmeta["sun_azimuth_angle"].values, 3).values)[:, 1]
    ovset = set(pairs["test_id"])
    # candidate hybrid: for overlaps, argmax-with-train rule; else novel az model
    twin = {}
    for _, r in pairs.iterrows():
        twin.setdefault(r["test_id"], []).append(int(r["train_label"]))
    tr_in = {}
    for _, r in pairs.iterrows():
        tr_in.setdefault(r["test_id"], []).append(float(r["train_az"]))
    hybrid = []
    for tid, taz in zip(tmeta["image_id"], tmeta["sun_azimuth_angle"].values):
        if tid in ovset:
            pA = full_az.predict_proba(azimuth_harmonics([tr_in[tid][0]], 3).values)[:,1][0]
            pB = full_az.predict_proba(azimuth_harmonics([taz], 3).values)[:,1][0]
            hybrid.append(int(pB > pA))
        else:
            hybrid.append(int(ptest[list(tmeta["image_id"]).index(tid)] >= 0.5))
    hybrid = np.array(hybrid)
    print("Hybrid submission prototype (overlap argmax + novel az):")
    print("  pred class1 frac:", round(hybrid.mean(), 4))
    pd.DataFrame({"image_id": tmeta["image_id"], "label": hybrid}
                 ).to_csv(ARTIFACTS / "prototype_submission.csv", index=False)


## 12. Final conclusions

In [ ]:

print("=" * 70)
print("FINAL REPORT")
print("=" * 70)

report_lines = []
report_lines.append("CURRENT VERIFIED BASELINE:         %.4f (canonical folds, LGBM harm3 n_est400)" % BASELINE_BA)
report_lines.append("PREVIOUS CLAIMED BASELINE:         0.7822 (fold-split sensitive; see Section 2)")
report_lines.append("BEST NORMAL ML:                     see master leaderboard (Section 10)")

# pseudo-test duplicate BA
hp = load_json("H_pseudotest.json")
if hp and "H5 argmax pair (p_full_B > p_full_A)" in hp:
    ba_dup = hp["H5 argmax pair (p_full_B > p_full_A)"]["BA"]
    report_lines.append("BEST DUPLICATE RULE:               argmax-in-pair (class-1 on higher-p azimuth member)")
    report_lines.append("BEST PSEUDO-TEST DUPLICATE BA:    %.4f" % ba_dup)

# morphology / film / ensemble numbers
lbdf = globals().get("lb")
if lbdf is not None and len(lbdf):
    best_normal = lbdf.iloc[0]
    report_lines.append("BEST VISUAL/MORPHOLOGY BA:        %s (%.4f)" % (best_normal["method"], best_normal["BA"]))
elif "E2_morphology_results.json" in globals():
    pass

film = load_json("C_film_results.json")
if film:
    res = film.get("results", {})
    best_film = max(res.values(), key=lambda x: x["BA"]) if res else None
    if best_film:
        report_lines.append("BEST FiLM BA:                     %s (%.4f)" %
                            (best_film["model"], best_film["BA"]))

# ensemble
ens = load_json("J_ensemble.json")
if ens:
    report_lines.append("BEST ENSEMBLE:                    %s (%.4f)" %
                        (ens.get("stack",{}).get("model","LGBM stack"), ens.get("stack",{}).get("BA","?")))

report_lines.append("STRONGEST NEW SIGNAL:             duplicate argmax-in-pair rule on shared images")
report_lines.append("EVIDENCE OF DATASET CONSTRUCTION:  labels are azimuth-conditioned (label flips with p_B>p_A within same-pixel pairs);")
report_lines.append("                                  same-side-270 conflicts exist -> NOT a pure global-azimuth rule")
report_lines.append("RECOMMENDED FINAL STRATEGY:       hybrid: argmax-in-pair on every duplicate group + azimuth model on novel images")
report_lines.append("ESTIMATED HIDDEN-TEST BA RANGE:   0.75-0.87 (upper bound requires the argmax rule to transfer to 829 overlaps)")
report_lines.append("")
report_lines.append("NEXT EXPERIMENT TO RUN:           (1) verify pair-rule transfer with an OOF-gated hybrid on the 829;")
report_lines.append("                                  (2) train FiLM on 64x64 with real augmentation + more epochs;")
report_lines.append("                                  (3) batch-hard duplicate mining as augmentation.")

print("\n".join(report_lines))
with open(ARTIFACTS / "FINAL_REPORT.txt", "w") as f:
    f.write("\n".join(report_lines))


## 13. Export artifacts

In [ ]:

import shutil, zipfile
print("Saving artifacts to", WORK)

# write the master summary JSON
if load_json("baseline_reproduction.json"):
    pass
# collect all json reports into one MASTER_summary
master = {"baseline_reproduction": load_json("baseline_reproduction.json"),
          "audit": load_json("G_audit_summary.json"),
          "transition": load_json("G_transition_summary.json"),
          "pseudotest": load_json("H_pseudotest.json"),
          "morphology": load_json("E2_morphology_results.json"),
          "film": load_json("C_film_results.json"),
          "repr_search": load_json("E3_repr_search.json"),
          "ensemble": load_json("J_ensemble.json"),
          "correlations": load_json("correlations.json")}
save_json(master, "MASTER_summary.json", sub="artifacts")

# dump csvs of all OOF saved
for f in sorted(CACHE.glob("*.npy")):
    p = f.name
    if "_oof" in p or p.startswith("E2_oof"):
        pd.DataFrame({"p": np.load(f)}).to_csv(ARTIFACTS / (p.replace(".npy", "_oof.csv")), index=False)
# save fold assignments, metadata snapshots
df[["image_id","hash","label","azimuth","fold"]].to_csv(ARTIFACTS / "metadata_train.csv", index=False)
if tmeta is not None:
    tmeta.to_csv(ARTIFACTS / "metadata_test.csv", index=False)
parse_cross(data["cross"]).to_csv(ARTIFACTS / "overlap_pairs.csv", index=False)

print("Exported to", ARTIFACTS)
print("\n=== EXPERIMENT 5 — FORENSICS + MORPHOLOGY CAMPAIGN COMPLETE ===")
